# MULTIPINN: установка и первый запуск

Этот notebook является **канонической инструкцией** по установке и запуску MULTIPINN.

Он согласован с `README.md` и должен использоваться как основная ссылка на туториал в репозитории и во внешних материалах.


## Что зафиксировано в этой инструкции

Чтобы в документации не было расхождений, в этом tutorial используются следующие единые правила:

- **Поддерживаемые версии Python:** `3.8` – `3.11`
- **Рекомендуемая версия для нового окружения:** `3.10`
- **Основная команда установки:** `pip install -e .`
- `make install` рассматривается только как короткая обёртка над той же командой

Ниже команды приводятся для запуска **из корня репозитория**.


## Что такое MULTIPINN

MULTIPINN — это фреймворк для решения задач математической физики с помощью **Physics-Informed Neural Networks (PINNs)**.  
Репозиторий содержит:

- библиотеку `multipinn/` с ядром фреймворка;
- готовые примеры в `examples/`;
- конфигурации на базе **Hydra**;
- средства логирования, сохранения моделей и визуализации результатов.


## Шаг 1. Клонирование репозитория

```bash
git clone https://github.com/multipinn/multipinn.git
cd multipinn
```


## Шаг 2. Создание и активация окружения

### Рекомендуемый вариант: `venv`

**Linux / macOS**
```bash
python3.10 -m venv .venv
source .venv/bin/activate
```

**Windows (PowerShell)**
```powershell
py -3.10 -m venv .venv
.\.venv\Scripts\Activate.ps1
```

Если у вас нет Python 3.10, используйте любую поддерживаемую версию из диапазона `3.8`–`3.11`.


### Альтернатива: `conda`

Если вы используете Conda, создайте окружение так:

```bash
conda create -n multipinn python=3.10
conda activate multipinn
```

Важно: способ создания окружения может отличаться, но **сама установка библиотеки ниже остаётся одинаковой**.


## Шаг 3. Установка MULTIPINN

Основная команда установки:

```bash
python -m pip install --upgrade pip
pip install -e .
```

Эквивалентная короткая команда из `Makefile`:

```bash
make install
```

Если для вашей машины нужен специальный CUDA-сборочный вариант PyTorch, сначала установите подходящий `torch`, а затем выполните `pip install -e .`.


## Шаг 4. Проверка установки

После установки должен успешно работать импорт библиотеки и PyTorch.


In [ ]:
import sys

print("Python:", sys.version.split()[0])

import torch
import multipinn

print("PyTorch:", torch.__version__)
print("MULTIPINN import OK")


## Шаг 5. Быстрый smoke-test

Для быстрой проверки удобно запустить короткий пример с уменьшенным числом эпох.  
Команда ниже запускает `regression_1D` и сохраняет интерактивные `html`-артефакты вместо `png`:

```bash
python -m examples.regression_1D.run_train \
  trainer.num_epochs=5 \
  generator.domain_points=256 \
  visualization.grid_plot_points=256 \
  visualization.save_period=10 \
  visualization.save_mode=html \
  paths.save_dir=./artifacts_smoke/regression_1D
```

Если команда выполняется без ошибок, установка библиотеки и базовый запуск настроены корректно.


## Шаг 6. Первый полноценный пример

Для более типичного PINN-сценария можно запустить задачу Пуассона:

```bash
python -m examples.poisson_2D_1C.run_train
```

Для более короткого локального прогона можно временно переопределить параметры Hydra:

```bash
python -m examples.poisson_2D_1C.run_train \
  trainer.num_epochs=100 \
  generator.domain_points=2000 \
  generator.bound_points=400 \
  visualization.grid_plot_points=1000 \
  visualization.save_period=100 \
  visualization.save_mode=html
```


## Где настраивать задачу

У большинства примеров структура одинаковая:

```text
examples/<example_name>/
├── configs/config.yaml
├── problem.py
└── run_train.py
```

Назначение файлов:

- `configs/config.yaml` — гиперпараметры, генераторы точек, пути сохранения, визуализация;
- `problem.py` — уравнения, геометрия, граничные и начальные условия;
- `run_train.py` — точка входа, которая создаёт модель, генераторы, callbacks и запускает обучение.


## Какие параметры меняют чаще всего

В `config.yaml` обычно настраиваются:

- `problem` — параметры физической постановки;
- `model` — архитектура сети;
- `regularization` — способ балансировки потерь;
- `generator` — число внутренних и граничных точек;
- `trainer` — число эпох, seed, частота обновления;
- `visualization` — плотность сетки, период сохранения, формат артефактов;
- `optimizer` и `scheduler` — оптимизатор и шаг обучения;
- `paths` — директория для результатов.

Это позволяет изменять большинство параметров **без правки исходного кода**.


## Где искать результаты

Результаты сохраняются в каталог, заданный в `paths.save_dir`.

Обычно там появляются:

- `used_config.yaml` — копия реально использованной конфигурации;
- веса модели (`.pth`);
- кривые обучения;
- визуализации (`html` и/или `png`);
- дополнительные файлы, создаваемые callbacks.

Если вам не нужны статические изображения, удобнее использовать:

```yaml
visualization:
  save_mode: html
```

Это снимает часть требований к локальной среде для PNG-экспорта.


## Важная заметка про GNN и mesh-примеры

Базовая установка тянет `torch_geometric` через `requirements.txt`.  
При этом часть graph- и mesh-сценариев может дополнительно требовать бинарные пакеты PyG, совместимые с вашей версией **PyTorch** и **CUDA**:

- `pyg_lib`
- `torch_scatter`
- `torch_sparse`
- `torch_cluster`
- `torch_spline_conv`

Если вы видите ошибки, связанные с `torch_scatter`, `torch_cluster`, `knn_graph` или `radius_graph`, проверьте версию Torch/CUDA:

```bash
python -c "import torch; print(torch.__version__, torch.version.cuda)"
```

После этого установите совместимые пакеты PyG и повторите запуск.


## Установка для разработки

Если вы не только запускаете примеры, но и дорабатываете библиотеку, используйте dev-установку:

```bash
pip install -e ".[dev]"
```

Эквивалентная короткая команда:

```bash
make install.dev
```

Полезные команды:

```bash
make install-pre-commit
make test
```


## Что считать канонической инструкцией

Чтобы в документации снова не возникло расхождений, при обновлении репозитория придерживайтесь следующего правила:

1. сначала обновляется этот notebook;
2. затем теми же формулировками синхронизируется `README.md`;
3. только после этого аналогичный текст переносится во внешние материалы, приложения к НТО или будущую Wiki-страницу.

Так все пользовательские инструкции будут ссылаться на одну и ту же актуальную версию tutorial.
